# HUGGE Alba — TRELLIS.2 PBR GLB

Этот notebook бесплатно запускает оптимизированный `trellis.cpp` на GPU Kaggle или Google Colab, загружает фотографию Alba с демонстрационного сайта MIRRAI и сохраняет текстурированную GLB-модель.

В Kaggle откройте **Settings → Accelerator → GPU P100** (или T4) и включите **Internet**. В Colab выберите **Runtime → Change runtime type → T4 GPU**. Затем нажмите **Run All**. Первый запуск скачивает около 9.5 ГБ весов Q8 и поэтому занимает заметное время.

In [ ]:
import os, pathlib, subprocess, time, requests

base = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path('/content')
work = base / 'trellis2'
work.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
install = f'curl -fsSL https://raw.githubusercontent.com/pwilkin/trellis.cpp/main/install/install.sh | bash -s -- --quant q8 --skip-app -y --dest {work}'
subprocess.run(['bash', '-lc', install], check=True)
print('TRELLIS.2 runtime and Q8 weights are ready')

In [ ]:
runtime = work / 'runtime'
server_bin = runtime / 'trellis-server'
models = work / 'models'
log_path = work / 'trellis-server.log'
env = os.environ.copy()
env['LD_LIBRARY_PATH'] = f"{runtime}:{env.get('LD_LIBRARY_PATH', '')}"
log = open(log_path, 'w')
server = subprocess.Popen([str(server_bin), '--models', str(models), '--host', '127.0.0.1', '--port', '8080', '--res', '1024', '--require-gpu'], stdout=log, stderr=subprocess.STDOUT, env=env)
for _ in range(180):
    try:
        if requests.get('http://127.0.0.1:8080/health', timeout=2).ok:
            break
    except requests.RequestException:
        pass
    if server.poll() is not None:
        raise RuntimeError(log_path.read_text()[-5000:])
    time.sleep(2)
else:
    raise TimeoutError('TRELLIS server did not become ready')
print('TRELLIS.2 server is ready')

In [ ]:
image_url = 'https://mirrai-try-on.moonlight-5782.chatgpt.site/catalog-sources/hugge-md/alba-89990-1.jpg'
image_path = work / 'alba-89990-1.jpg'
image_response = requests.get(image_url, timeout=60)
image_response.raise_for_status()
image_path.write_bytes(image_response.content)
output = base / 'alba-trellis2-pbr.glb'
with image_path.open('rb') as source:
    response = requests.post(
        'http://127.0.0.1:8080/generate',
        files={'image': ('alba.jpg', source, 'image/jpeg')},
        data={'seed': '89990', 'resolution': '1024', 'bg_removal': 'birefnet'},
        timeout=3600,
    )
response.raise_for_status()
output.write_bytes(response.content)
print(f'Готово: {output} ({output.stat().st_size / 1024 / 1024:.1f} MB)')
from IPython.display import FileLink, display
display(FileLink(str(output)))
try:
    from google.colab import files
    files.download(str(output))
except ImportError:
    pass